# 01. Разведочный анализ данных

Датасет — [MIT Indoor Scenes](https://web.mit.edu/torralba/www/indoor.html), из 67 категорий взяты 5 жилых комнат.
Разбиение на train/val/test (70/15/15, стратифицировано по классам) делается скриптом `python -m src.prepare_data`,
здесь только анализ результата.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

from src.config import CLASSES, PROCESSED_DATA_DIR

SPLITS = ["train", "val", "test"]

## Распределение классов

In [ ]:
counts = pd.DataFrame(
    {split: [len(list((PROCESSED_DATA_DIR / split / c).glob("*.jpg"))) for c in CLASSES] for split in SPLITS},
    index=CLASSES,
)
counts.loc["total"] = counts.sum()
counts

In [ ]:
ax = counts.drop("total").plot.bar(figsize=(9, 4), rot=0)
ax.set_title("Число изображений по классам")
ax.set_ylabel("Изображений")
plt.tight_layout()
plt.show()

**Вывод.** Классы несбалансированы: `kitchen` в train почти в 4 раза больше, чем `bathroom` (513 против 137).
Поэтому кроме accuracy смотрим на **macro-F1** — она одинаково взвешивает все классы и не даёт
модели «выехать» на частых кухнях и гостиных.

## Примеры изображений

In [ ]:
n_per_class = 5
fig, axes = plt.subplots(len(CLASSES), n_per_class, figsize=(3 * n_per_class, 3 * len(CLASSES)))
for row, class_name in zip(axes, CLASSES):
    paths = sorted((PROCESSED_DATA_DIR / "train" / class_name).glob("*.jpg"))[:n_per_class]
    for ax, path in zip(row, paths):
        ax.imshow(Image.open(path).convert("RGB"))
        ax.axis("off")
    row[0].set_title(class_name, loc="left", fontsize=14)
plt.tight_layout()
plt.show()

**Наблюдения.** `dining_room` и `livingroom` часто сняты в open-space планировках, где в кадр попадает
и стол, и диван, — это ожидаемо самая трудная пара для классификатора. Ванные и кухни визуально
самые «узнаваемые» (сантехника, кухонный гарнитур).

## Размеры и формат изображений

In [ ]:
records = []
for split in SPLITS:
    for path in (PROCESSED_DATA_DIR / split).rglob("*.jpg"):
        with Image.open(path) as img:
            records.append({"split": split, "class": path.parent.name,
                            "width": img.width, "height": img.height, "mode": img.mode})
sizes = pd.DataFrame(records)
sizes["aspect"] = sizes["width"] / sizes["height"]
sizes["min_side"] = sizes[["width", "height"]].min(axis=1)

print(sizes["mode"].value_counts().to_string(), "\n")
print("Меньше 224 px по короткой стороне:", (sizes["min_side"] < 224).sum())
sizes[["width", "height", "aspect"]].describe().round(2)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
sizes["min_side"].clip(upper=1000).hist(bins=50, ax=ax1)
ax1.set_title("Короткая сторона, px (обрезано на 1000)")
sizes["aspect"].hist(bins=50, ax=ax2)
ax2.set_title("Соотношение сторон (width / height)")
plt.tight_layout()
plt.show()

**Выводы для препроцессинга.**
- Размеры сильно разные (от ~200 px до 3000+ px), медиана — 256×256. Поэтому все изображения приводятся
  к одному виду: `Resize(256)` по короткой стороне + `CenterCrop(224)` — стандартный вход ResNet.
- Соотношение сторон — от ~0.5 до ~2.9. Центральный кроп у очень вытянутых панорам отрезает края,
  на обучении это частично компенсирует `RandomResizedCrop`.
- Несколько изображений в режимах `L` (grayscale) и `P` (палитра) — `ImageFolder` и API
  конвертируют всё в RGB, иначе нормализация на 3 канала упадёт.
- Около десятка картинок меньше 224 px — они будут немного апскейлены, на качество это почти не влияет.